# 🚀 파이토치 기초 실습 (Google Colab용)

## 0. Google Colab GPU 활성화
상단 메뉴: 런타임 → **런타임 유형 변경** → 하드웨어 가속기를 **'GPU' 또는 'T4 GPU'로 선택**

## 1. 환경 설정 및 임포트
코랩은 기본적으로 파이토치가 설치되어 있지만, 버전을 확인하고 필요한 라이브러리를 불러옵니다.

In [3]:
import torch

print(f"PyTorch Version: {torch.__version__}")

# PyTorch CUDA 사용 가능 여부 확인
print(torch.cuda.is_available())    # True여야 함
print(torch.cuda.get_device_name(0))    # GPU 이름 출력

PyTorch Version: 2.10.0+cu128
True
Tesla T4


In [4]:
import torch.nn as nn
import torch.optim as optim
import torchvision
import numpy as np

## 2. PyTorch 프레임워크 소개
### 왜 파이토치인가?
*   직관적인 문법 : Python과 유사한 구조로 학습 곡선이 낮음
*   동적 그래프 : 실행 시점에 그래프 생성, 디버깅 용이
* 강력한 커뮤니티 : 최신 논문 대부분이 PyTorch 기반
* 산업 표준 : YOLOv8, Transformer 등 최신 모델 지원



## 3. 텐서(Tensor) 기본기
* PyTorch의 기본 데이터 구조
* Numpy의 Ndarray와 유사하지만 GPU 가속이 가능

In [5]:
# 리스트로부터 텐서 생성
data = [[1, 2, 3], [4, 5, 6]]
x = torch.tensor(data)
print("x =", x)

x = tensor([[1, 2, 3],
        [4, 5, 6]])


In [7]:
# 1. 기본 연산
print("x + 1 =", x + 1)
print("x * 2 =", x * 2)

# 2. 브로드캐스팅
b = torch.tensor([1, 0, -1])
print("x + [b] =", x + b)

# 3. 자동미분(AutoGrad) 테스트
a = torch.tensor(3.0, requires_grad=True) # 텐서 a = 3.0, 이 변수(a)와 관련된 모든 연산을 추적한 후, 미분값을 계산 // 미분을 사용 하겠다.
b = a**2 + 2*a + 1  # 수식 정의 : a를 기반으로 미분을 하겠다.
b.backward()        # backpropagation : 출력(b)에서 입력(a) 방향으로 거꾸로 가며 미분(기울기)을 계산, db/da(b를 a로 미분)를 구하는 과정
# 미분 이용해서 거꾸로 적용하겠다. -> 미분으로 되어있는 requires_grad를 적용.
#Pytorch 는 requires_grad랑 backward만 있어도 쉽게 계산 o.
print("db/da =", a.grad)

x + 1 = tensor([[2, 3, 4],
        [5, 6, 7]])
x * 2 = tensor([[ 2,  4,  6],
        [ 8, 10, 12]])
x + [b] = tensor([[2, 2, 2],
        [5, 5, 5]])
db/da = tensor(8.)


### 이미지 텐서 형태 : (B, C, H, W)
* B(Batch)
* C(Channel)
* H(Height)
* W(width)

In [8]:
import cv2

apple_path = '/content/apple.jpg' #현재 상태가 /content // openCV로 불러옴.

In [9]:
# GRAY Apple 이미지 로드
gray_apple_img = cv2.imread(apple_path, cv2.IMREAD_GRAYSCALE) #처음에는 빨간색 사과인데, gray 사과 이미지로 읽어들임.

if gray_apple_img is not None:
    # NumPy 배열을 PyTorch 텐서로 변환
    # H x W -> 1 x H x W (채널 추가, 배치 차원 없음)
    gray_apple_tensor = torch.from_numpy(gray_apple_img).float().unsqueeze(0).unsqueeze(0) # 두 개의 차원 추가(B,C)
    #읽어들인 이미지를 numpy로//  unsqueeze : 텐서에 크기 1인 차원을 추가하는 함수

    print(f"Apple Image Loaded: {apple_path}")
    print(f"Original NumPy Shape (H, W): {gray_apple_img.shape}")
    print(f"Tensor Shape (B, C, H, W): {gray_apple_tensor.shape}")
    print(f"Tensor Shape (C, H, W): {gray_apple_tensor.squeeze(0).shape}") #squeeze: 차원 줄임.
    print(f"Tensor Data Type: {gray_apple_tensor.dtype}")
    print(f"Tensor Device: {gray_apple_tensor.device}") # PyTorch는 기본적으로 텐서를 CPU에 생성
                                                        # 명시적으로 텐서나 모델을 GPU로 옮겨주는 작업 필요
                                                        # Pytorch는 CPU로 기본적. 그래서 GPU로 바꿔줘야함.
else:
    print(f"Error: Could not load image from {apple_path}")

Apple Image Loaded: /content/apple.jpg
Original NumPy Shape (H, W): (199, 253)
Tensor Shape (B, C, H, W): torch.Size([1, 1, 199, 253])
Tensor Shape (C, H, W): torch.Size([1, 199, 253])
Tensor Data Type: torch.float32
Tensor Device: cpu


In [11]:
# COLOR Apple 이미지 로드
color_apple_img = cv2.imread(apple_path)

if color_apple_img is not None:
    # NumPy 배열을 PyTorch 텐서로 변환
    # H x W -> 1 x H x W (채널 추가, 배치 차원 없음)
    color_apple_tensor = torch.from_numpy(color_apple_img).float().permute(2, 0, 1).unsqueeze(0) # 한 개의 차원 추가(B) # permute: 차원의 순서 바꾸겠다.
    #permute(2, 0, 1)는 0번째 차원(H), 1번째 차원(W), 2번째 차원(C)을 각각 새로운 0번째, 1번째, 2번째 차원으로 재배치하라는 의미

    print(f"Color Apple Image Loaded: {apple_path}")
    print(f"Original NumPy Shape (H, W, C): {color_apple_img.shape}") # Numpy그대로 쓸 수 x. 이미지 tensor로 만들어줘야함. # H는 0차원, W는 1차원, C는 2차원.
    print(f"Tensor Shape (B, C, H, W): {color_apple_tensor.shape}")  #B추가해줌.
    print(f"Tensor Shape (C, H, W): {color_apple_tensor.squeeze(0).shape}")  #squeeze하면 가장 앞에꺼 지워짐.
    print(f"Tensor Data Type: {color_apple_tensor.dtype}")
    print(f"Tensor Device: {color_apple_tensor.device}") # PyTorch는 기본적으로 텐서를 CPU에 생성
                                                        # 명시적으로 텐서나 모델을 GPU로 옮겨주는 작업 필요
else:
    print(f"Error: Could not load image from {apple_path}")

Color Apple Image Loaded: /content/apple.jpg
Original NumPy Shape (H, W, C): (199, 253, 3)
Tensor Shape (B, C, H, W): torch.Size([1, 3, 199, 253])
Tensor Shape (C, H, W): torch.Size([3, 199, 253])
Tensor Data Type: torch.float32
Tensor Device: cpu
